In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

groq_key = os.environ.get("GROQ_KEY")


In [2]:
groq_llm = ChatGroq(api_key=groq_key, model="llama-3.3-70b-versatile")

In [5]:
from langchain_core.messages import HumanMessage, AIMessage

In [4]:
groq_llm.invoke([HumanMessage(content="Hello World!")])

AIMessage(content="Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 38, 'total_tokens': 63, 'completion_time': 0.053264075, 'completion_tokens_details': None, 'prompt_time': 0.000952156, 'prompt_tokens_details': None, 'queue_time': 0.048845814, 'total_time': 0.054216231}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c7b8a-2f94-7f70-b4a9-1a3bc7642fd7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 25, 'total_tokens': 63})

In [9]:
groq_llm.invoke([
    HumanMessage(content="Hello World!, fuck you"),
    AIMessage(
        content="Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?"),
    HumanMessage(content="what is my previous msg and replay to that"),
]).content

'Your previous message was: "Hello World!, fuck you"\n\nAnd my response to that is: I see you\'re expressing some strong emotions. I\'m here to listen and help if you\'d like to talk about what\'s on your mind. Please know that I\'m a neutral and non-judgmental space for you to express yourself. Would you like to discuss what\'s bothering you?'

### Message history

In [11]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [20]:
store = {}


def get_session_history(session_id) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


config = {
    'configurable': {
        "session_id": "chat1"
    }
}

In [21]:
print(store)
print(get_session_history(1))
print(get_session_history(2))
print(store)

{}


{1: InMemoryChatMessageHistory(messages=[]), 2: InMemoryChatMessageHistory(messages=[])}


In [22]:
with_chat_history = RunnableWithMessageHistory(groq_llm, get_session_history)

In [24]:
with_chat_history.invoke([
    HumanMessage(content="Hello World!, fuck you"),
    AIMessage(content="Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?"),
    HumanMessage(content="what is my previous msg and replay to that"),
],config=config)

AIMessage(content='Your previous message was: "Hello World!, fuck you"\n\nAnd my response to that is: I understand that you\'re expressing strong emotions, but I want to focus on the "Hello World!" part. It\'s a phrase often used to introduce people to programming and technology. If you\'re interested in learning more about coding or tech, I\'d be happy to help. Alternatively, if you\'d like to talk about what\'s bothering you, I\'m here to listen and provide support.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 232, 'total_tokens': 330, 'completion_time': 0.345641191, 'completion_tokens_details': None, 'prompt_time': 0.014922845, 'prompt_tokens_details': None, 'queue_time': 0.049562885, 'total_time': 0.360564036}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c7b98-90c2-7c62-9251-7dd5

In [67]:
for chat in store['chat1'].messages:
    print(chat.response_metadata)

AttributeError: 'dict' object has no attribute 'completion_tokens'

In [38]:
print(with_chat_history.invoke([
    HumanMessage(content="who am i"),
],config=config).content)

You are the person who initiated this chat with me. You've been interacting with me through a series of messages, starting with "Hello World!, fuck you". Throughout our conversation, you've been asking me to recall your previous messages and respond to them.

As for your identity, I don't have any personal information about you, such as your name, location, or background. You're anonymous to me, and our conversation is a text-based interaction. If you'd like to share more about yourself or discuss a specific topic, I'm here to listen and help!
